# Validación: macro vs. medición manual

Para ver si la macro da lo mismo que mi medición manual y si sale igual al repetir el trazo.

Columnas de `plantilla_validacion.csv` (una fila por estructura):

| columna | qué va |
|---|---|
| `celula` | id de la estructura |
| `variable` | `Largo` o `Area` |
| `manual` | medición manual |
| `macro` | medición con `flagelos.ijm` |
| `macro_repeticion` | opcional, la misma estructura trazada otra vez (otro día u otra persona) |

Mínimo unas 20 a 30 estructuras, de chicas a grandes.

Se calcula:
1. Bland-Altman: sesgo y límites de concordancia al 95 %.
2. CCC de Lin: qué tanto caen los puntos sobre la línea de identidad.
3. ICC(2,1): repetibilidad entre la primera y la segunda medición con la macro.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ARCHIVO = Path("plantilla_validacion.csv")
datos = pd.read_csv(ARCHIVO)
if datos.empty:
    raise SystemExit("Plantilla vacía, faltan los datos.")
datos.head()

## Funciones

In [ ]:
def bland_altman(a, b):
    """Sesgo y LoA 95 % de b contra a."""
    dif = b - a
    sesgo, de = dif.mean(), dif.std(ddof=1)
    return sesgo, sesgo - 1.96 * de, sesgo + 1.96 * de


def ccc_lin(a, b):
    """CCC de Lin."""
    ma, mb = a.mean(), b.mean()
    va, vb = a.var(ddof=1), b.var(ddof=1)
    cov = np.cov(a, b, ddof=1)[0, 1]
    return 2 * cov / (va + vb + (ma - mb) ** 2)


def icc_2_1(x):
    """ICC(2,1), acuerdo absoluto. x: n estructuras x k mediciones."""
    x = np.asarray(x, float)
    n, k = x.shape
    gm = x.mean()
    ssr = k * ((x.mean(1) - gm) ** 2).sum()
    ssc = n * ((x.mean(0) - gm) ** 2).sum()
    sse = ((x - gm) ** 2).sum() - ssr - ssc
    msr, msc, mse = ssr / (n - 1), ssc / (k - 1), sse / ((n - 1) * (k - 1))
    return (msr - mse) / (msr + (k - 1) * mse + k * (msc - mse) / n)

## Concordancia macro vs. manual

In [ ]:
filas = []
for var, d in datos.groupby("variable"):
    d = d.dropna(subset=["manual", "macro"])
    sesgo, li, ls = bland_altman(d["manual"], d["macro"])
    r = stats.pearsonr(d["manual"], d["macro"])[0]
    fila = {"variable": var, "n": len(d),
            "sesgo": sesgo, "sesgo_%": 100 * sesgo / d["manual"].mean(),
            "LoA_inf": li, "LoA_sup": ls, "r_Pearson": r,
            "CCC_Lin": ccc_lin(d["manual"], d["macro"])}
    rep = d.dropna(subset=["macro_repeticion"])
    if len(rep) >= 3:
        fila["ICC(2,1)_repetibilidad"] = icc_2_1(rep[["macro", "macro_repeticion"]])
    filas.append(fila)
tabla = pd.DataFrame(filas).round(3)
tabla.to_csv("resumen_validacion.csv", index=False)
tabla

## Gráficas

In [ ]:
variables = list(datos["variable"].unique())
fig, ejes = plt.subplots(len(variables), 2, figsize=(10, 4.2 * len(variables)), squeeze=False)
for (ax1, ax2), var in zip(ejes, variables):
    d = datos[datos["variable"] == var].dropna(subset=["manual", "macro"])
    a, b = d["manual"], d["macro"]

    # dispersión + identidad
    lim = [min(a.min(), b.min()) * 0.95, max(a.max(), b.max()) * 1.05]
    ax1.plot(lim, lim, "--", color="gray", lw=1, label="identidad")
    ax1.scatter(a, b, s=28, alpha=0.8)
    ax1.set(xlim=lim, ylim=lim, xlabel=f"{var} manual", ylabel=f"{var} macro",
            title=f"{var}: CCC = {ccc_lin(a, b):.3f}")
    ax1.legend(frameon=False)

    # Bland-Altman
    sesgo, li, ls = bland_altman(a, b)
    media = (a + b) / 2
    ax2.scatter(media, b - a, s=28, alpha=0.8)
    for y, txt in [(sesgo, "sesgo"), (li, "−1.96 DE"), (ls, "+1.96 DE")]:
        ax2.axhline(y, ls="-" if txt == "sesgo" else "--", color="gray", lw=1)
        ax2.annotate(f"{txt}: {y:.2f}", (media.max(), y), ha="right", va="bottom", fontsize=8)
    ax2.set(xlabel="media (manual, macro)", ylabel="macro − manual", title=f"{var}: Bland-Altman")

fig.tight_layout()
fig.savefig("../docs/img/validacion.png", dpi=150)
plt.show()

## Para el reporte

> La concordancia entre `flagelos.ijm` y la medición manual se evaluó en *n* estructuras con Bland-Altman (sesgo = …; LoA 95 % = … a …) y el CCC de Lin (CCC = …). La repetibilidad de la macro fue ICC(2,1) = ….

La figura se guarda en `docs/img/validacion.png`.